# Compare feature extractors × classifier heads (head to head)

Flexible bench for the nucleus classifier. It auto-discovers every curated
feature set under `data/classifier/*.npz` (e.g. `uni2h_fold2`,
`dinov3_vitl16_fold2`, `dinov3_vitb16_fold2`) and every classifier head you list,
then compares them on **identical splits**:

- macro-F1 table + heatmap over **encoder × head**
- per-class F1 by encoder
- confusion matrices side by side
- embedding spaces side by side (UMAP/t-SNE by class)
- silhouette / kNN purity by encoder

Two dicts at the top (`FEATURE_SETS`, `HEADS`) control everything — add a curated
`.npz` or a new sklearn head and re-run.

> For a fair comparison, curate each encoder with the **same** `--fold --n --seed
> --max-per-class` so the nuclei (and splits) align row-for-row.


## Config — edit these

In [ ]:
from pathlib import Path
import glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from vlm_medseg.classify.dataset import CropDataset

def _root():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "classifier").is_dir():
            return b
    return Path.cwd()

# Auto-discover curated feature sets (override this dict to pick a subset).
FEATURE_SETS = {Path(p).stem: p for p in sorted(glob.glob(str(_root() / "data/classifier/*.npz")))}

# Classifier heads to compare (add your own; each is a fresh sklearn estimator).
HEADS = {
    "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced")),
    "svm_rbf": lambda: make_pipeline(StandardScaler(), SVC(C=1.0, kernel="rbf", class_weight="balanced")),
    "knn15": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "mlp": lambda: make_pipeline(StandardScaler(), MLPClassifier(
        hidden_layer_sizes=(512,), alpha=1e-3, max_iter=500, early_stopping=True, random_state=0)),
}
# NOTE: sklearn's MLPClassifier has no class_weight, so (unlike logreg/svm) it is
# NOT balanced -> expect a lower Dead F1. If MLP looks promising, the balanced
# upgrade is a small torch MLP head with focal / class-balanced loss.
print("feature sets:", list(FEATURE_SETS))
print("heads:", list(HEADS))

## Load feature sets

In [ ]:
datasets = {}
for name, path in FEATURE_SETS.items():
    try:
        datasets[name] = CropDataset.load(path)
    except Exception as e:
        print("skip", name, "->", e)

assert datasets, "no feature sets found — run scripts/curate_classifier_data.py first"
CLS = next(iter(datasets.values())).class_names
NC = len(CLS)
for name, d in datasets.items():
    from collections import Counter
    print(f"{name:24s} n={d.features.shape[0]:5d} dim={d.features.shape[1]:5d} "
          f"encoder={d.encoder:14s} splits={dict(Counter(d.split.tolist()))}")

## Encoder × head — accuracy & macro-F1

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

rows, fitted = [], {}
for enc, d in datasets.items():
    X, y, split = d.features, d.class_id, d.split
    tr = split == "train"
    for head, factory in HEADS.items():
        clf = factory().fit(X[tr], y[tr])
        fitted[(enc, head)] = clf
        for sp in ("val", "test"):
            m = split == sp
            if not m.any():
                continue
            pred = clf.predict(X[m])
            rows.append({"encoder": enc, "head": head, "split": sp,
                         "acc": accuracy_score(y[m], pred),
                         "macroF1": f1_score(y[m], pred, average="macro", labels=range(NC), zero_division=0)})
res = pd.DataFrame(rows)
test = res[res.split == "test"].pivot(index="encoder", columns="head", values="macroF1")
print("TEST macro-F1 (encoder x head):")
test.round(3)

In [ ]:
# heatmap of test macro-F1
fig, ax = plt.subplots(figsize=(1.6 * len(HEADS) + 2, 0.7 * len(datasets) + 1.5))
im = ax.imshow(test.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(test.shape[1])); ax.set_xticklabels(test.columns)
ax.set_yticks(range(test.shape[0])); ax.set_yticklabels(test.index)
for i in range(test.shape[0]):
    for j in range(test.shape[1]):
        ax.text(j, i, f"{test.values[i,j]:.2f}", ha="center", va="center",
                color="white" if test.values[i,j] < 0.6 else "black")
ax.set_title("test macro-F1  ·  encoder × head"); fig.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.show()

## Per-class F1 by encoder (selected head)
Defaults to the **MLP** head; set `HEAD` to any key in `HEADS` to switch.

In [ ]:
HEAD = "mlp" if "mlp" in HEADS else ("logreg" if "logreg" in HEADS else list(HEADS)[0])
print("selected head:", HEAD)
per = {}
for enc, d in datasets.items():
    X, y, split = d.features, d.class_id, d.split
    te = split == "test"
    pred = fitted[(enc, HEAD)].predict(X[te])
    per[enc] = f1_score(y[te], pred, average=None, labels=range(NC), zero_division=0)

x = np.arange(NC); w = 0.8 / max(1, len(per))
fig, ax = plt.subplots(figsize=(1.6 * NC + 2, 4))
for i, (enc, f1) in enumerate(per.items()):
    ax.bar(x + i * w, f1, w, label=enc)
ax.set_xticks(x + w * (len(per) - 1) / 2); ax.set_xticklabels(CLS, rotation=20)
ax.set_ylim(0, 1); ax.set_ylabel(f"test F1 ({HEAD})"); ax.set_title("per-class F1 by encoder")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Confusion matrices side by side (selected `HEAD`)

In [ ]:
n = len(datasets)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4))
axes = np.atleast_1d(axes)
for ax, (enc, d) in zip(axes, datasets.items()):
    X, y, split = d.features, d.class_id, d.split
    te = split == "test"
    cm = confusion_matrix(y[te], fitted[(enc, HEAD)].predict(X[te]), labels=range(NC))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NC)); ax.set_yticks(range(NC))
    ax.set_xticklabels(CLS, rotation=45, ha="right", fontsize=7); ax.set_yticklabels(CLS, fontsize=7)
    acc = np.trace(cm) / max(1, cm.sum())
    ax.set_title(f"{enc}\nacc={acc:.2f}", fontsize=9); ax.set_xlabel("pred"); ax.set_ylabel("gt")
    for i in range(NC):
        for j in range(NC):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()

## Embedding spaces side by side (by class)
Standardize → PCA(50) → UMAP (t-SNE fallback), one panel per encoder.

In [ ]:
from sklearn.decomposition import PCA
from vlm_medseg.constants import CLASS_COLORS
try:
    import umap; _proj = lambda Z: ("UMAP", umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Z))
except Exception:
    from sklearn.manifold import TSNE; _proj = lambda Z: ("t-SNE", TSNE(n_components=2, init="pca", random_state=0).fit_transform(Z))

n = len(datasets)
fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5))
axes = np.atleast_1d(axes)
for ax, (enc, d) in zip(axes, datasets.items()):
    Xs = StandardScaler().fit_transform(d.features)
    Z = PCA(n_components=min(50, Xs.shape[1]), random_state=0).fit_transform(Xs)
    name, emb = _proj(Z)
    for c in range(NC):
        m = d.class_id == c
        ax.scatter(emb[m, 0], emb[m, 1], s=6, alpha=0.6, color=np.array(CLASS_COLORS[c]) / 255, label=CLS[c])
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"{enc} · {name}")
axes[0].legend(markerscale=2, fontsize=7)
plt.tight_layout(); plt.show()

## Embedding quality by encoder (silhouette · kNN purity)

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

q = []
for enc, d in datasets.items():
    Xs = StandardScaler().fit_transform(d.features); y = d.class_id
    sil = silhouette_score(Xs, y, sample_size=min(3000, len(y)), random_state=0)
    _, idx = NearestNeighbors(n_neighbors=6).fit(Xs).kneighbors(Xs)
    purity = np.mean([(y[idx[i, 1:]] == y[i]).mean() for i in range(len(y))])
    q.append({"encoder": enc, "silhouette": float(sil), "kNN5_purity": float(purity)})
qdf = pd.DataFrame(q).set_index("encoder")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
qdf["silhouette"].plot.bar(ax=ax[0], color="#4c72b0", title="silhouette (by class)"); ax[0].tick_params(axis="x", rotation=20)
qdf["kNN5_purity"].plot.bar(ax=ax[1], color="#55a868", title="kNN(5) label purity"); ax[1].set_ylim(0, 1); ax[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()
qdf.round(3)

## Takeaways
- The **encoder × head heatmap** is the headline: rows = feature extractors, columns = classifier heads. Read down a column to compare encoders under the same head; across a row to see how much head choice matters.
- **Per-class F1 + confusion** show *where* an encoder wins/loses (usually the subtle Connective/Epithelial and rare Dead).
- **Embedding spaces + silhouette/kNN purity** explain the table: a model that clusters classes more tightly will probe better, largely independent of the head.
- Domain (UNI2-h, H&E-pretrained) vs scale/recency (DINOv3 ViT-L/B, general) shows up here directly — let the numbers decide.
